# Notebook 2 — QAOA on the simulator

Run QAOA on our 5-node test graph at depths $p=1, 2, 3$ using Qiskit Aer's statevector simulator. We track the approximation ratio as a function of $p$ and visualize the measurement distribution.

In [ ]:
import sys; sys.path.insert(0, '../src')
from max_cut import MaxCut
from qaoa import run_qaoa, build_qaoa_circuit
from plotting import setup_matplotlib, plot_measurement_distribution
import matplotlib.pyplot as plt

setup_matplotlib()

mc = MaxCut.from_edges(5, [(0,1),(1,2),(2,0),(1,3),(3,4),(4,0)], name='5-node test')
opt_cut, opt_z = mc.brute_force_optimal()
print(f'Optimal cut: {opt_cut} (bitstring {opt_z})')

## Run QAOA at depths p = 1, 2, 3

In [ ]:
results = {}
for p in [1, 2, 3]:
    print(f'\n=== QAOA at p = {p} ===')
    results[p] = run_qaoa(mc, depth=p, n_shots=4096, max_iter=200, seed=123)

## Measurement distribution at p=3 (most probable outcomes)

In [ ]:
plot_measurement_distribution(results[3].counts, mc, '../figures/measurement_distribution.png')
plt.show()

## Approximation ratio vs p

The headline result: QAOA at $p=3$ exceeds the Goemans-Williamson ratio (0.878) on this graph.

In [ ]:
from plotting import plot_approximation_ratio
p_vals = [1, 2, 3]
ratios = [results[p].approximation_ratio for p in p_vals]
plot_approximation_ratio(p_vals, ratios, '../figures/approx_ratio_vs_p.png')
plt.show()
print(f'\nApproximation ratios: p=1: {ratios[0]:.3f}, p=2: {ratios[1]:.3f}, p=3: {ratios[2]:.3f}')
print(f'GW ratio: 0.878 (theoretical lower bound)')

## Save the QAOA circuit diagrams

In [ ]:
from plotting import plot_qaoa_circuit
for p in [1, 2, 3]:
    qc = build_qaoa_circuit(mc, results[p].params[:p].tolist(), results[p].params[p:].tolist())
    plot_qaoa_circuit(qc, f'../figures/qaoa_circuit_p{p}.png')
    print(f'Saved circuit for p={p}')